# Conditional Probability experiment¶

In this notebook we make a wild experiment exploring how far we could go using conditional probabilities. In a nutshell:

* we take all numeric features and split them into buckets
* use Theil's U as categorical feature selector
* measure the conditional probability of our target class when being in a certain bucket (for each feature)
* we average these probabilities out
* we take the softmax to scale the output
* we do this in multiple iterations and take a final average.

Disclaimer: This is highly experimental and not expected to add any value for your rankings in this competition. Any ideas to improve this script are welcome. There might also be bugs left.


# Load the data

In [ ]:
import numpy as np
import pandas as pd
import re
from typing import Optional, Tuple, Union

import matplotlib.pyplot as plt
from xgboost import plot_tree

import warnings
warnings.filterwarnings("ignore")


from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import PowerTransformer, LabelEncoder

In [ ]:
train = pd.read_csv('/kaggle/input/credit-card-fraud-prediction/train.csv')
test = pd.read_csv('/kaggle/input/credit-card-fraud-prediction/test.csv')
submission = pd.read_csv('/kaggle/input/credit-card-fraud-prediction/sample_submission.csv')

#cols = df_orig.columns
#train = pd.concat((train, df_orig), axis=0).reset_index(drop=True)

print('The dimension of the train dataset is:', train.shape)
print('The dimension of the test dataset is:', test.shape)

In [ ]:
train

In [ ]:
test

In [ ]:
train.info()

In [ ]:
train.describe()

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# Feature engineering

In this section we will build features using various techniques.

In [ ]:
target = "IsFraud"

In [ ]:
train[target].value_counts()

In [ ]:
# add binary features based on shap dependency plots
def add_binary_flags(df: pd.DataFrame) -> pd.DataFrame:
    df["feat9_feat6_binary"] = ((df["feat9"] > 0) & (df["feat6"] > 1))
    df["feat4_feat13_binary"] = ((df["feat4"] > 0) & (df["feat13"] > 0.5))
    df["Transaction_Amount_feat9_binary"] = ((df["Transaction_Amount"] > 0) & (df["feat13"] > 1.2))
    df["feat18_feat19_binary"] = ((df["feat18"] > 0) & (df["feat19"] <= 0.5) & (df["feat19"] >= -0.5))
    df["feat18_feat20_binary"] = ((df["feat18"] > 0) & (df["feat20"] <= 0.5) & (df["feat20"] >= -0.5))
    return df

In [ ]:
train = add_binary_flags(train)
test = add_binary_flags(test)

In [ ]:
train = train.drop("id", axis=1)
test = test.drop("id", axis=1)

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# ProbaBoost

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1" # export OMP_NUM_THREADS=1
os.environ["OPENBLAS_NUM_THREADS"] = "1" # export OPENBLAS_NUM_THREADS=1
os.environ["MKL_NUM_THREADS"] = "1" # export MKL_NUM_THREADS=1
os.environ["VECLIB_MAXIMUM_THREADS"] = "1" # export VECLIB_MAXIMUM_THREADS=1
os.environ["NUMEXPR_NUM_THREADS"] = "1" # export NUMEXPR_NUM_THREADS=1

from sklearn import preprocessing
import numpy as np
import gc
from collections import Counter
import pandas as pd
import scipy.stats as ss
from numpy import dot
from numpy.linalg import norm
import math
from sklearn.metrics import matthews_corrcoef
from joblib import Parallel, delayed
import optuna



from sklearn import preprocessing
import numpy as np
import gc
from collections import Counter
import pandas as pd
import scipy.stats as ss
from numpy import dot
from numpy.linalg import norm
import math


class ConditionProbabilitiesClassifier:
    """
    This classifier measures the conditional probability of the target variable for each feature
    separately. Numerical values are converted into buckets for this purpose.
    Additionally, the explanatory power of each feature is measured using Theil's U. The conditional probabilities
    are then weighted by Theil's U.
    :param df: Pandas DataFrame containing the target feature.
    :param cat_feats: List of strings with column names specifying categorical features.
    :param num_feats: List of strings with column names specifying numerical features.
    :param target: String specifying name of the target column.
    """

    def __init__(self,
                 df: pd.DataFrame,
                 cat_feats: list,
                 num_feats: list,
                 target: str,
                 nb_buckets: int = 5):
        self.df = df.reset_index(drop=True)
        self.cat_feats = cat_feats
        self.num_feats = num_feats
        self.target = target
        self.nb_buckets = nb_buckets
        
        self.thresholds = np.linspace(0, 1, self.nb_buckets).tolist()
        self.thresholds.append(2000000)
        self.num_buckets = [f"< {threshold}" for threshold in self.thresholds]
        self.cat_cond_probs = {}
        self.num_cond_probs = {}
        self.cat_theil_u_weights = {}
        self.num_theil_u_weights = {}
        self.cat_selected_features = []
        self.num_selected_features = []
        self.scalers = {}

    def fill_nulls(self, df: pd.DataFrame, col: str, fill_with=None, coltype: str = 'str'):
        """
        Fill nulls with provided term.
        :param df: Pandas DaaFrame
        :param col: String specifying column name.
        :param fill_with: String, integer or float to fill missing values with.
        :param coltype: One of 'str' or 'float'. Will be used to typecast the column.
        """
        if coltype == 'str':
            df[col] = df[col].astype(str)
        elif coltype == 'float':
            df[col] = df[col].astype(np.float64)

        df[col] = df[col].fillna(fill_with)
        return df

    def conditional_entropy(self, x, y):
        # entropy of x given y
        y_counter = Counter(y)
        xy_counter = Counter(list(zip(x, y)))
        total_occurrences = sum(y_counter.values())
        entropy = 0
        for xy in xy_counter.keys():
            p_xy = xy_counter[xy] / total_occurrences
            p_y = y_counter[xy[1]] / total_occurrences
            entropy += p_xy * math.log(p_y / p_xy)
        return entropy

    def theil_u(self, x, y):
        s_xy = self.conditional_entropy(x, y)
        x_counter = Counter(x)
        total_occurrences = sum(x_counter.values())
        p_x = list(map(lambda n: n / total_occurrences, x_counter.values()))
        s_x = ss.entropy(p_x)
        if s_x == 0:
            return 1
        else:
            return (s_x - s_xy) / s_x

    def fit_theil_u_weights(self):
        """
        Loops through categorical and numerical columns and measured their target explainability
        using Theil's U.
        :return: Updates class attributes.
        """
        for col in self.cat_feats:
            theil_u = self.theil_u(self.df[col], self.df[self.target])
            self.cat_theil_u_weights[col] = theil_u

        for col in self.num_feats:
            theil_u = self.theil_u(self.df[col], self.df[self.target])
            self.num_theil_u_weights[col] = theil_u

    def fit_theil_u_feature_selection(self):
        """
        Loops through all columns and selects columns with Theil U value of 0.01 or higher.
        :return: Updates selected_features attribute.
        """
        for col in self.cat_feats:
            if self.cat_theil_u_weights[col] >= 0.00:
                self.cat_selected_features.append(col)

        for col in self.num_feats:
            if self.num_theil_u_weights[col] >= 0.00:
                self.num_selected_features.append(col)

    def fit_categorical_cond_probas(self):
        """
        Loops through all categorical features and their categories
        and maps the conditional probability of the target.
        :return: Updates class attributes.
        """
        for col in self.cat_feats:
            self.cat_cond_probs[col] = {}
            for category in self.df[col].unique():
                temp_df = self.df.loc[self.df[col] == category]
                cond_prob = (temp_df[self.target].sum() / len(self.df.index)) / (  # joint prob
                        len(temp_df.index) / len(self.df.index))  # marginal
                self.cat_cond_probs[col][category] = cond_prob

            if "Unknown" not in self.cat_cond_probs[col].keys():
                self.cat_cond_probs[col]["Unknown"] = 0

    def fit_predict_buckets(self, col: str):
        """
        Takes a column of a numerical type and returns it as a bucketed version.
        :param col: String specifying which column to scale and return as bucketed version.
        :return: Pandas DataFrame
        """
        # scale Series
        scaler = preprocessing.MinMaxScaler()
        scaled = scaler.fit_transform(self.df[[col]])
        scaled = pd.DataFrame(scaled, columns=[col])
        self.scalers[col] = scaler
        
        # add random noise to also occupy highest bucket
        scaled += 0.001

        # transform into buckets
        conditions = []
        for thres in self.thresholds:
            conditions.append((scaled[col] < thres))
        
        choices = self.num_buckets

        scaled[col] = np.select(conditions, choices, default='Unknown')
        return scaled

    def predict_buckets(self, df, col: str):
        """
        Takes a Pandas Series of a numerical type and returns it as a bucketed version.
        :param col: String specifying which column to scale and return as bucketed version.
        returns: Pandas DataFrame
        """
        # scale Series
        scaler = self.scalers[col]
        scaled = scaler.transform(df[[col]])
        scaled = pd.DataFrame(scaled, columns=[col])

        # transform into buckets
        conditions = []
        for thres in self.thresholds:
            conditions.append((scaled[col] < thres))

        choices = self.num_buckets

        scaled[col] = np.select(conditions, choices, default='Unknown')
        return scaled

    def fit_numerical_cond_probas(self):
        """
        Loops through all numerical features and their buckets
        and maps the conditional probabilities of the target.
        """
        for col in self.num_feats:
            bucketed = self.fit_predict_buckets(col=col)
            bucketed[self.target] = self.df[self.target]
            self.num_cond_probs[col] = {}
            cond_prob = 0
            for bucket in self.num_buckets:
                temp_df = bucketed.loc[bucketed[col] == bucket]
                if temp_df.empty:
                    pass
                else:
                    cond_prob = (temp_df[self.target].sum() / len(self.df.index)) / (  # joint prob
                            len(temp_df.index) / len(self.df.index))  # marginal
                self.num_cond_probs[col][bucket] = cond_prob

            if "Unknown" not in self.num_cond_probs[col].keys():
                self.num_cond_probs[col]["Unknown"] = 0

            del bucketed
            _ = gc.collect()
            
    def softmax(self, x):
        """Compute softmax values for each sets of scores in x."""
        return np.exp(x) / np.sum(x ** 2.1, axis=0)  # softmax is adjusted here to improve the balanced logloss

    def fit(self):
        """
        Wrapper function to fit conditional probabilities
        of categorial and numerical features. Updates the
        dictionaries self.cat_cond_probs and self.num_cond_probs.
        """
        for col in self.cat_feats:
            self.df = self.fill_nulls(self.df, col, fill_with='Unknown', coltype='str')

        for col in self.num_feats:
            self.df = self.fill_nulls(self.df, col, fill_with=0, coltype='float')
            self.df = self.df.replace([np.inf, -np.inf], 0)

        self.fit_categorical_cond_probas()
        self.fit_numerical_cond_probas()
        self.fit_theil_u_weights()
        self.fit_theil_u_feature_selection()

    def fit_predict(self):
        """
        Wrapper function to fit conditional probabilities
        of categorial and numerical features. Updates the
        dictionaries self.cat_cond_probs and self.num_cond_probs.
        """
        self.fit()

        # transform cols into conditional probas
        for col in self.cat_selected_features:
            for category in self.df[col].unique():
                if category not in self.cat_cond_probs[col].keys():
                    self.cat_cond_probs[col][category] = 0
            self.df = self.df.replace(self.cat_cond_probs[col])
            #self.df[col] = self.df[col] * self.cat_theil_u_weights[col]

        for col in self.num_selected_features:
            self.df[col] = self.predict_buckets(self.df, col)
            self.df = self.df.replace(self.num_cond_probs[col])
            #self.df[col] = self.df[col] * self.num_theil_u_weights[col]

        # get predictions
        preds = self.df[self.cat_selected_features + self.num_selected_features].mean(axis=1)
        self.df["preds_class_1"] = preds
        self.df["preds_class_0"] = 1 - preds
        preds = pd.DataFrame(self.softmax(self.df[["preds_class_1", "preds_class_0"]].values), columns=["preds_class_1", "preds_class_0"])
        return preds["preds_class_1"]

    def predict(self, df: pd.DataFrame):
        """
        Create predictions based on conditional probabilities and their Theil's U weights.
        :param df: Pandas DataFrame for prediction.
        :return Pandas Series
        """
        # transform cols into conditional probas
        for col in self.cat_selected_features:
            df = self.fill_nulls(df, col, fill_with='Unknown')
            for category in df[col].unique():
                if category not in self.cat_cond_probs[col].keys():
                    self.cat_cond_probs[col][category] = 0
            df = df.replace(self.cat_cond_probs[col])
            #df[col] = df[col] * self.cat_theil_u_weights[col]

        for col in self.num_selected_features:
            df = self.fill_nulls(df, col, fill_with=0, coltype='float')
            df = df.replace([np.inf, -np.inf], 0)
            df[col] = self.predict_buckets(df, col)
            df = df.replace(self.num_cond_probs[col])
            #df[col] = df[col] * self.num_theil_u_weights[col]

        # get predictions
        preds = df[self.cat_selected_features + self.num_selected_features].mean(axis=1)
        df["preds_class_1"] = preds
        df["preds_class_0"] = 1 - preds
        self.df = df
        preds = pd.DataFrame(self.softmax(df[["preds_class_1", "preds_class_0"]].values), columns=["preds_class_1", "preds_class_0"])
        return preds["preds_class_1"]


class ProbaBoost:
    """
    This classifier measures the conditional probability of the target variable for each feature
    separately. Numerical values are converted into buckets for this purpose.
    Additionally, the explanatory power of each feature is measured using Theil's U. The conditional probabilities
    are then weighted by Theil's U.
    :param df: Pandas DataFrame containing the target feature.
    :param cat_feats: List of strings with column names specifying categorical features.
    :param num_feats: List of strings with column names specifying numerical features.
    :param target: String specifying name of the target column.
    """

    def __init__(self,
                 df: pd.DataFrame,
                 cat_feats: list,
                 num_feats: list,
                 target: str,
                 nb_boosters: int = 30,
                 nb_buckets: int = 20,
                 random_state=5000):
        self.df = df
        self.cat_feats = cat_feats
        self.num_feats = num_feats
        self.target = target
        self.nb_boosters = nb_boosters
        self.nb_buckets = nb_buckets
        self.random_state = random_state
        self.conditional_classifiers = []

    def train_model(self, i, tcss):      
        
        sample_df = self.df.sample(tcss, random_state=self.random_state+i, replace=True)
        sample_df = sample_df.reset_index(drop=True)

        cpc = ConditionProbabilitiesClassifier(
            df=sample_df,
            cat_feats=self.cat_feats,
            num_feats=self.num_feats,
            target=self.target,
            nb_buckets=self.nb_buckets)

        cpc.fit()
        return {"model_obj": cpc,
                "nb_buckets": self.nb_buckets}

    def model_predict(self, i, df):
        df = df.reset_index(drop=True)
        cpc = self.conditional_classifiers[i]
        churn_proba = cpc.predict(df)
        del cpc
        _ = gc.collect()
        preds = {}
        preds[i] = churn_proba
        return preds

    def fit(self):
        model_objects = Parallel(n_jobs=-1)(
            delayed(self.train_model)(i, len(self.df.index)) for i in range(self.nb_boosters))

        for obj in model_objects:
            if len(obj["model_obj"].cat_feats + obj["model_obj"].num_feats) > 0:
                self.conditional_classifiers.append(obj["model_obj"])

        del model_objects
        _ = gc.collect()

    def predict(self, df):       
        prediction_objects = Parallel(n_jobs=-1)(
            delayed(self.model_predict)(i, df) for i in range(self.nb_boosters))

        predictions = {}
        for i in prediction_objects:
            for key, item in i.items():
                predictions[key] = item

        del prediction_objects
        _ = gc.collect()

        pred_df = pd.DataFrame(predictions)


        pred_df["predictions"] = pred_df.mean(axis=1)
        pred_df["predictions"] = pred_df["predictions"].astype(np.float64)
        return pred_df["predictions"]

In [ ]:
num_cols = train.drop(target, axis=1).columns.to_list()

cbd = ProbaBoost(
        df = train.fillna(0),
        cat_feats=[],
        num_feats=num_cols,
        target=target,
        nb_boosters=30,
        nb_buckets=10
    )
    
cbd.fit()

In [ ]:
fraud_cases = cbd.predict(test.fillna(0))

# Submission time

In [ ]:
submission[target] = fraud_cases
submission.to_csv('submission.csv', index=False)

In [ ]:
submission

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>